# Lezione 38 — Transfer learning e freezing

> **Modulo:** LoRA e adattamento efficiente · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 11 (dentro il training), Lezione 32 (blocco Transformer).
>
> Obiettivo pratico unico: capire perche' si **congelano** (freeze) i pesi
> pre-addestrati e si addestra solo una piccola parte — la premessa che rende
> sensato LoRA (Lezione 39).

## Teoria essenziale

Un modello pre-addestrato (Lezione 34) ha gia' imparato rappresentazioni utili
da enormi corpora. Per adattarlo a un compito nuovo hai due estremi:

- **Full fine-tuning**: aggiorni *tutti* i pesi. Massima flessibilita', ma costo
  e memoria enormi, e rischio di dimenticare cio' che il modello sapeva
  (*catastrophic forgetting*).
- **Freezing + testa nuova**: **congeli** il corpo pre-addestrato (i suoi
  gradienti non vengono calcolati ne' applicati) e addestri solo una piccola
  testa nuova. Economico, ma a volte poco espressivo.

LoRA (Lezione 39) sta nel mezzo: congela il corpo *e* aggiunge pochi parametri
addestrabili dentro gli strati. Oggi vediamo la meccanica base del freezing.

In [1]:
import numpy as np

rng = np.random.default_rng(38)

# corpo "pre-addestrato" (finto): due strati densi. Lo trattiamo come dato.
W1 = rng.normal(size=(8, 16)) * 0.3
W2 = rng.normal(size=(16, 16)) * 0.3

# testa nuova, addestrabile, per il nostro compito (3 tipi di memoria)
W_head = rng.normal(size=(16, 3)) * 0.3

def conta(*matrici):
    return sum(m.size for m in matrici)

tot = conta(W1, W2, W_head)
addestrabili = conta(W_head)          # solo la testa
print(f"parametri totali      : {tot}")
print(f"parametri addestrabili: {addestrabili}  ({100*addestrabili/tot:.1f}%)")
print(f"parametri congelati   : {tot - addestrabili}")

parametri totali      : 432
parametri addestrabili: 48  (11.1%)
parametri congelati   : 384


Leggi i numeri: congelando il corpo, addestriamo una frazione minima dei
parametri. Il **gradiente** si calcola e si applica *solo* ai pesi addestrabili;
per i congelati non si fa il passo di aggiornamento.

In [2]:
# dimostrazione: un passo di training che aggiorna SOLO la testa
def forward(x, W1, W2, W_head):
    h = np.maximum(0, x @ W1)      # ReLU
    h = np.maximum(0, h @ W2)
    return h @ W_head              # logit

x = rng.normal(size=(4, 8))        # 4 esempi
target = np.array([0, 1, 2, 1])

W1_prima, W2_prima = W1.copy(), W2.copy()

# gradiente numerico semplice solo su W_head (gli altri restano identici)
def perdita(W_head):
    logit = forward(x, W1, W2, W_head)
    logit = logit - logit.max(axis=1, keepdims=True)
    logp = logit - np.log(np.exp(logit).sum(axis=1, keepdims=True))
    return -logp[np.arange(len(target)), target].mean()

eps = 1e-4
grad = np.zeros_like(W_head)
base = perdita(W_head)
for i in range(W_head.shape[0]):
    for j in range(W_head.shape[1]):
        W_head[i, j] += eps
        grad[i, j] = (perdita(W_head) - base) / eps
        W_head[i, j] -= eps
W_head = W_head - 0.5 * grad        # aggiorno solo la testa

# il corpo congelato NON e' cambiato
assert np.array_equal(W1, W1_prima) and np.array_equal(W2, W2_prima)
print("perdita prima:", round(base, 4), "-> dopo:", round(perdita(W_head), 4))
print("corpo congelato invariato:", np.array_equal(W1, W1_prima))

perdita prima: 1.2423 -> dopo: 0.9098
corpo congelato invariato: True


## Il progetto: Memory AI Lab, passo 38

Fisso una piccola utility che, dato un insieme di "moduli", calcola quanti
parametri sono addestrabili sotto una politica di freezing — cosi' nelle
prossime lezioni possiamo confrontare full fine-tuning, freezing e LoRA con
numeri coerenti.

In [3]:
def riepilogo_parametri(moduli, addestrabili_keys):
    tot = sum(m.size for m in moduli.values())
    train = sum(moduli[k].size for k in addestrabili_keys)
    return {"totali": tot, "addestrabili": train,
            "percentuale": round(100 * train / tot, 2)}

moduli = {"W1": W1, "W2": W2, "W_head": W_head}
r = riepilogo_parametri(moduli, ["W_head"])
assert r["addestrabili"] == W_head.size
assert r["addestrabili"] < r["totali"]
print("freezing (solo testa):", r)

freezing (solo testa): {'totali': 432, 'addestrabili': 48, 'percentuale': 11.11}


## Riepilogo (max 8 punti)

1. Un modello pre-addestrato porta rappresentazioni gia' utili.
2. **Full fine-tuning**: aggiorni tutto → costoso, rischio di *forgetting*.
3. **Freezing**: congeli il corpo, addestri solo una testa piccola → economico.
4. Congelare = non calcolare/applicare il gradiente a quei pesi.
5. Il corpo congelato resta bit-identico dopo il training.
6. Freezing puro puo' essere poco espressivo.
7. LoRA (Lezione 39) sta nel mezzo: corpo congelato + pochi pesi *dentro* gli
   strati.
8. Contare i parametri addestrabili e' il metro per confrontare le strategie.

## Quiz

1. Cosa significa "congelare" un peso, in termini di gradiente?
2. Perche' il full fine-tuning rischia il catastrophic forgetting?
3. Qual e' il limite del solo freezing + testa nuova?

*(Risposte: 1. non calcolare/applicare il suo gradiente, resta invariato;
2. aggiornando tutti i pesi si sovrascrive cio' che il modello aveva imparato;
3. la testa aggiunta puo' non bastare a esprimere il nuovo compito se il corpo
resta fisso.)*